## 1. Lab 5 - Dynamic Web Scraping and Data Cleaning
 - Source Website
 - Dynamic Data Collection
 - Missing Values
 - String Cleaning
 - Data Validation
 - Final Dataset
 - Findings

In [67]:
%pip install selenium
%pip install -q "pandera[pandas]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 535.0/535.0 kB 11.5 MB/s eta 0:00:0000:01


In [68]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandera.pandas as pa

import pandas as pd

In [27]:
options = webdriver.ChromeOptions()

options.add_argument("--headless=new")  # Run in headless mode (no GUI)
options.add_argument("--disable-gpu")  # Disable GPU acceleration
options.add_argument("--no-sandbox")  # Bypass OS security model
options.add_argument("--disable-dev-shm-usage")  # Overcome limited resource problems
options.add_argument("--window-size=1920,1080")  # Set window size to avoid issues with responsive design


In [39]:
import subprocess

def available_package(*names):
    for name in names:
        result = subprocess.run(
            ["apt-cache", "policy", name],
            capture_output=True, text=True
        )
        if any(
            "Candidate:" in line and "(none)" not in line
            for line in result.stdout.splitlines()
        ):
            return name
    raise RuntimeError(f"No available package among: {names}")

packages = [
    available_package("libatk-bridge2.0-0t64", "libatk-bridge2.0-0"),
    available_package("libxcomposite1"),
    available_package("libatspi2.0-0t64", "libatspi2.0-0"),
]

subprocess.run(["apt-get", "install", "-y", *packages], check=True)

CompletedProcess(args=['apt-get', 'install', '-y', 'libatk-bridge2.0-0t64', 'libxcomposite1', 'libatspi2.0-0t64'], returncode=0)

In [41]:
driver = webdriver.Chrome(options=options)


In [42]:
url = "https://quotes.toscrape.com/js/"
driver.get(url)

In [43]:
WebDriverWait(driver, 10).until(
    EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, ".quote")
    )
)

[<selenium.webdriver.remote.webelement.WebElement (session="52bcedff1cc52d397ea804dacbc3b952", element="f.97C72C896D2066E123D301FE908C917A.d.3B11DB2E6DE094303ACDB7C6AF0CD185.e.137")>,
 <selenium.webdriver.remote.webelement.WebElement (session="52bcedff1cc52d397ea804dacbc3b952", element="f.97C72C896D2066E123D301FE908C917A.d.3B11DB2E6DE094303ACDB7C6AF0CD185.e.138")>,
 <selenium.webdriver.remote.webelement.WebElement (session="52bcedff1cc52d397ea804dacbc3b952", element="f.97C72C896D2066E123D301FE908C917A.d.3B11DB2E6DE094303ACDB7C6AF0CD185.e.5")>,
 <selenium.webdriver.remote.webelement.WebElement (session="52bcedff1cc52d397ea804dacbc3b952", element="f.97C72C896D2066E123D301FE908C917A.d.3B11DB2E6DE094303ACDB7C6AF0CD185.e.139")>,
 <selenium.webdriver.remote.webelement.WebElement (session="52bcedff1cc52d397ea804dacbc3b952", element="f.97C72C896D2066E123D301FE908C917A.d.3B11DB2E6DE094303ACDB7C6AF0CD185.e.140")>,
 <selenium.webdriver.remote.webelement.WebElement (session="52bcedff1cc52d397ea804

In [44]:
quotes = driver.find_elements(
    By.CSS_SELECTOR,
    ".quote"
)

In [45]:
len(quotes)

10

In [46]:
card = quotes[0]

In [47]:
card.find_element(
    By.CSS_SELECTOR,
    ".text"
).text

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [48]:
card.find_element(
    By.CSS_SELECTOR,
    ".author"
).text

'Albert Einstein'

In [49]:
tags = card.find_elements(
    By.CSS_SELECTOR,
    ".tag"
)

In [50]:
[t.text for t in tags]

['change', 'deep-thoughts', 'thinking', 'world']

In [59]:
rows = []
pages = 0
wait = WebDriverWait(driver, 20)

while True:
    wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, ".quote .text")))
    cards = driver.find_elements(By.CSS_SELECTOR, ".quote")

    for card in cards:
        quote = card.find_element(By.CSS_SELECTOR, ".text").text
        author = card.find_element(By.CSS_SELECTOR, ".author").text
        tags = card.find_elements(By.CSS_SELECTOR, ".tag")
        tags = ", ".join(tag.text for tag in tags)
        rows.append({"quote": quote, "author": author, "tags": tags})

    pages += 1
    next_button = driver.find_elements(By.CSS_SELECTOR, ".next a")
    if not next_button:
        break

    next_button[0].click()
    wait.until(EC.staleness_of(cards[0]))

driver.quit()

In [61]:
raw_df = pd.DataFrame(rows, columns=["quote", "author", "tags"])

print("Pages:", pages)
print("Quotes:", len(raw_df))
raw_df.head()

Pages: 10
Quotes: 100


,quote,author,tags
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational"


In [62]:
df = raw_df.replace(r"^\s*$", pd.NA, regex=True).copy()
missing_before = df.isna().sum()

df = df.dropna(subset=["quote", "author"])
df["tags"] = df["tags"].fillna("untagged")

missing_after = df.isna().sum()
pd.DataFrame({"Before": missing_before, "After": missing_after})

,Before,After
quote,0,0
author,0,0
tags,3,0


In [63]:
for column in ["quote", "author", "tags"]:
    df[column] = df[column].str.replace(r"\s+", " ", regex=True).str.strip()

df["quote"] = df["quote"].str.strip('“”"')
df["tags"] = df["tags"].str.lower()
df["tags"] = df["tags"].str.replace(r"\s*,\s*", ", ", regex=True)

df["tag_count"] = df["tags"].str.count(",") + 1
df.loc[df["tags"] == "untagged", "tag_count"] = 0

changed_quotes = (raw_df.loc[df.index, "quote"] != df["quote"]).sum()
duplicates = df.duplicated(subset=["quote", "author"]).sum()
df = df.drop_duplicates(subset=["quote", "author"]).reset_index(drop=True)
df.head()

,quote,author,tags,tag_count
0,The world as we have created it is a process o...,Albert Einstein,"change, deep-thoughts, thinking, world",4
1,"It is our choices, Harry, that show what we tr...",J.K. Rowling,"abilities, choices",2
2,There are only two ways to live your life. One...,Albert Einstein,"inspirational, life, live, miracle, miracles",5
3,"The person, be it gentleman or lady, who has n...",Jane Austen,"aliteracy, books, classic, humor",4
4,"Imperfection is beauty, madness is genius and ...",Marilyn Monroe,"be-yourself, inspirational",2


In [69]:
schema = pa.DataFrameSchema({
    "quote": pa.Column(str, pa.Check.str_length(min_value=1), nullable=False),
    "author": pa.Column(str, pa.Check.str_length(min_value=1), nullable=False),
    "tags": pa.Column(str, pa.Check.str_length(min_value=1), nullable=False),
    "tag_count": pa.Column(int, pa.Check.ge(0), nullable=False),
}, unique=["quote", "author"], coerce=True)

clean_df = schema.validate(df)
print("Validation passed")

Validation passed


In [70]:
print("Missing values:")
print(clean_df.isna().sum())
print("Duplicates:", clean_df.duplicated(subset=["quote", "author"]).sum())
print("Data types:")
print(clean_df.dtypes)

raw_df.to_csv("quotes_raw.csv", index=False)
clean_df.to_csv("quotes_cleaned.csv", index=False)
clean_df

Missing values:
quote        0
author       0
tags         0
tag_count    0
dtype: int64
Duplicates: 0
Data types:
quote        object
author       object
tags         object
tag_count     int64
dtype: object


,quote,author,tags,tag_count
0,The world as we have created it is a process o...,Albert Einstein,"change, deep-thoughts, thinking, world",4
1,"It is our choices, Harry, that show what we tr...",J.K. Rowling,"abilities, choices",2
2,There are only two ways to live your life. One...,Albert Einstein,"inspirational, life, live, miracle, miracles",5
3,"The person, be it gentleman or lady, who has n...",Jane Austen,"aliteracy, books, classic, humor",4
4,"Imperfection is beauty, madness is genius and ...",Marilyn Monroe,"be-yourself, inspirational",2
...,...,...,...,...
95,You never really understand a person until you...,Harper Lee,better-life-empathy,1
96,You have to write the book that wants to be wr...,Madeleine L'Engle,"books, children, difficult, grown-ups, write, ...",7
97,Never tell the truth to people who are not wor...,Mark Twain,truth,1
98,"A person's a person, no matter how small.",Dr. Seuss,inspirational,1


In [71]:
print(f"1. Collected {len(raw_df)} quotes from {pages} pages.")
print(f"2. Missing cells: {missing_before.sum()} before handling and {missing_after.sum()} after.")
print(f"3. Cleaned whitespace or enclosing quotation marks in {changed_quotes} quotes; standardized tags.")
print(f"4. Removed {duplicates} duplicate quote-author pairs; {len(clean_df)} records remain.")
print("5. Added tag counts and passed Pandera validation for all four columns.")

1. Collected 100 quotes from 10 pages.
2. Missing cells: 3 before handling and 0 after.
3. Cleaned whitespace or enclosing quotation marks in 100 quotes; standardized tags.
4. Removed 0 duplicate quote-author pairs; 100 records remain.
5. Added tag counts and passed Pandera validation for all four columns.
